# Earnings Brief Agent — Chained Multi-turn Research

## Overview

This notebook builds a financial research agent that, given a stock ticker, automatically searches for analyst estimates, recent competitor results, and sector news — then generates a structured pre-earnings call brief.

**Why live search is critical here**: Analyst consensus, guidance revisions, and competitor earnings are filed days before the call. This information does not exist in any model's training data.

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Agentic framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4 |
| Tutorial vertical | Finance |
| Example complexity | Intermediate |
| SDK used | boto3, strands-agents |

### Chained Search Pattern

```
Input: Ticker (e.g., "AMZN")
         │
         ▼
Search 1: Analyst EPS + revenue estimates
         │  results inform → what consensus expects
         ▼
Search 2: Guidance revisions since last quarter
         │  results inform → any surprise risk
         ▼
Search 3: Top competitors' recent earnings
         │  results inform → sector context
         ▼
Search 4: Macro / sector news this quarter
         │
         ▼
Output: Structured pre-earnings brief
```

## Prerequisites

Complete `01-gateway-setup-and-raw-mcp` first and export the environment variables it prints.

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": "bedrock:InvokeModel",
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configuration and Setup

In [ ]:
import os
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
GATEWAY_URL = os.environ["AGENTCORE_GATEWAY_URL"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["COGNITO_CLIENT_ID"]
CLIENT_SECRET = os.environ["COGNITO_CLIENT_SECRET"]
SCOPE_STRING = os.environ["COGNITO_SCOPE"]
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"


def get_token():
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET, "scope": SCOPE_STRING}
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def create_transport():
    return streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {get_token()}"})


print("Configuration loaded ✓")

## 3. Choose a Ticker

In [ ]:
# Change this to any US-listed company ticker
TICKER = "AMZN"
COMPANY_NAME = "Amazon"  # Used in search queries for better results

print(f"Generating pre-earnings brief for: {TICKER} ({COMPANY_NAME})")

## 4. Define the Earnings Brief Agent

The system prompt instructs the agent to follow the four-search research sequence and produce a structured output.

In [ ]:
EARNINGS_SYSTEM_PROMPT = """You are a financial research analyst with access to real-time web search.

When given a stock ticker and company name, produce a pre-earnings call brief by:

RESEARCH SEQUENCE (do these searches in order):
1. Search: "<ticker> earnings estimate analyst consensus Q<current quarter> 2026"
2. Search: "<company> guidance revision outlook 2026"
3. Search: "<company> competitors earnings results recent"
4. Search: "<company> sector macro headwinds tailwinds 2026"

CRITICAL RULES:
- Keep each search query under 200 characters
- Each search informs the next — note what you learned before issuing the next query
- Cite URLs for every claim

OUTPUT FORMAT (use markdown):

## Pre-Earnings Brief: <TICKER> — <Quarter> <Year>

### Analyst Consensus
<EPS estimate, revenue estimate, beat/miss probability>

### Guidance and Surprise Risk
<any recent guidance changes, analyst upgrades/downgrades>

### Competitor Context
<how key competitors performed recently>

### Macro and Sector Factors
<external tailwinds and headwinds this quarter>

### What to Watch
<3 key metrics or statements to monitor during the call>

### Sources
<list of URLs cited>
"""

mcp_client = MCPClient(create_transport)
print("Earnings brief agent ready ✓")

## 5. Generate the Pre-Earnings Brief

The agent will run 4 chained searches — each one informed by the previous result — then synthesize the brief.

In [ ]:
brief_prompt = f"""Generate a pre-earnings call brief for {TICKER} ({COMPANY_NAME}).

Follow the research sequence: analyst estimates → guidance revisions → competitor context → macro factors.
Produce the full structured brief with cited sources.
"""

print(f"Researching {TICKER}...")
print("-" * 60)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.5, max_tokens=3000)
    agent = Agent(model=model, tools=tools, system_prompt=EARNINGS_SYSTEM_PROMPT)
    response = agent(brief_prompt)

print("\n" + "=" * 60)
print(f"PRE-EARNINGS BRIEF: {TICKER}")
print("=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## 6. Try Another Ticker

In [ ]:
# Change these to generate a brief for a different company
TICKER_2 = "TSLA"
COMPANY_2 = "Tesla"

brief_prompt_2 = f"""Generate a pre-earnings call brief for {TICKER_2} ({COMPANY_2}).
Follow the research sequence and produce the full structured brief with cited sources.
"""

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.5, max_tokens=3000)
    agent = Agent(model=model, tools=tools, system_prompt=EARNINGS_SYSTEM_PROMPT)
    response = agent(brief_prompt_2)

print(f"\n{'=' * 60}")
print(f"PRE-EARNINGS BRIEF: {TICKER_2}")
print("=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## Conclusion

In this notebook you built a financial research agent that:
- Runs 4 chained searches where each result shapes the next query
- Synthesizes analyst data, competitive context, and macro factors
- Produces a structured brief with cited sources

The key insight: **the output of each search is used to form a better next query** — this is the chained multi-turn pattern.

**Next**: See `03-iterative-research.ipynb` for the generalized search → reflect → search loop that underpins both this example and the CVE scanner.